# The Cellular Interface, Multicellular

_Investigation `the-cellular-interface-multicellular` — coder reproduction notebook._

**Question.** The companion investigation `draft-to-living-cell` specified the paper's nine
interface-composition patterns and ran them with toy-real, lumped mechanisms (ODEs, scripted
events, 1-D or 9-cell grids). This investigation asks whether the same patterns hold when the
environment is not a lumped store but a real 2D spatial field, and the cell is not a scalar
state bundle but a Cellular Potts Model (CPM) agent with genuine shape, footprint, and
motion. Concretely: does the Fig 5 sense/act loop survive being realized as a real CPM cell
(viva-cpm) sitting in a spatio-flux diffusing nutrient field, running dynamic flux-balance
analysis at its own footprint, growing, and secreting a metabolic byproduct back into the
field it depends on — composed from two independently-developed simulation frameworks through
one typed coupling interface (`CpmCellField`)?

This investigation is the spatial sibling of `draft-to-living-cell`. That
investigation turned the cellular-interface paper's nine composition patterns into inert
typed drafts compiled into small, lumped executables (ODE-scale mechanisms on 1-D or 9-cell
grids). This investigation asks the harder question: do those same patterns still compose
when the frameworks on each side of the interface are real, independently-built 2D spatial
simulators — a Cellular Potts Model (viva-cpm, a Rust-backed lattice engine with genuine cell
shape, adhesion, and volume dynamics) on one side, and spatio-flux's diffusion-advection PDE
solver plus dynamic flux-balance analysis on the other? Composing two frameworks that were
never designed with each other in mind is exactly the paper's central claim about typed
interfaces made concrete: `CpmCellField`, the one coupling process this investigation has
built so far, is the interface where a CPM cell's footprint (a set of lattice pixels) is
translated into a spatio-flux field read, and where a dynamic-FBA solution (mmol/gDW/hr
fluxes) is translated back into a per-pixel field delta and a CPM target-volume update. Only
the flagship study, `cell-environment-coupling-spatial` (the spatial analogue of Fig 5's
sense/act loop), is built and run; the other eight patterns from `draft-to-living-cell`
(cell-cell coupling, disintegration, molecular interfaces, biomolecular complementarity,
autopoiesis, growth-and-division, development-and-evolution, and the cellular-interface
contract itself) are named as future increments in the design spec but have no composite,
study, or run yet.

The investigations are deliberate analogues, study-for-study, wherever the counterpart
exists: `cell-environment-coupling-spatial` here is the spatial realization of
`draft-to-living-cell/cell-environment-coupling`. Read them side by side — the lumped 9-cell
1-D executable there against the real 60x60 2D CPM+spatio-flux composite here — for the same
interface claim (sensing and acting are one coupling, not two subsystems; the cell measurably
reshapes the gradient it depends on) demonstrated at two very different levels of mechanistic
resolution.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/meta-modelers-guide/meta-modelers-guide').is_dir():
    REPO = Path('/home/runner/work/meta-modelers-guide/meta-modelers-guide')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from meta_modelers_guide.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Cell–Environment Coupling, Spatial (flagship) (`cell-environment-coupling-spatial`)

**Question.** Does the Fig 5 sense/act loop hold as real spatial metabolism — one CPM cell sensing
a diffusing nutrient field, running flux-balance analysis at its own footprint, growing, and
secreting a byproduct back into the field — when the cell and its environment are each real,
independently-built simulators (viva-cpm's Cellular Potts engine, spatio-flux's
diffusion-advection and dynamic-FBA processes) composed through one typed coupling process
(`CpmCellField`), rather than a scripted or lumped stand-in for that coupling?

**Claim.** Composing a real CPM cell with a real spatio-flux nutrient field through one typed
coupling process (`CpmCellField`) reproduces the Fig 5 sense/act loop as genuine spatial
metabolism, not a scripted stand-in for it: over 20 ticks the cell runs dynamic-FBA at its own
footprint (`e_coli_core`, O2-capped to force acetate overflow), grows ~3.4x in CPM volume (32
-> 110 px, ~3% of a 60x60 lattice it does not come close to filling), depletes local glucose by
up to ~18% before diffusion partially resupplies it (niche construction), and secretes a
diffusing acetate plume — two independently-built simulators talking through one typed port.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `single-cell-in-a-field` | `meta_modelers_guide.composites.single-cell-in-a-field` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.single-cell-in-a-field`** — `spec_meta_modelers_guide_composites_single_cell_in_a_field` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_single_cell_in_a_field = load_spec(REPO / 'meta_modelers_guide/composites/single-cell-in-a-field.composite.json')
describe_spec(spec_meta_modelers_guide_composites_single_cell_in_a_field)

In [ ]:
# === Edit parameters for composite 'single-cell-in-a-field' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'diff'  (local:!spatio_flux.processes.diffusion_advection.DiffusionAdvection)
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['diff']['config']['n_bins'] = [60, 60]
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['diff']['config']['bounds'] = [60.0, 60.0]
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['diff']['config']['diffusion_coeffs']['glucose'] = 0.4
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['diff']['config']['diffusion_coeffs']['acetate'] = 0.6
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['diff']['config']['boundary_conditions']['glucose']['default']['type'] = 'neumann'
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['diff']['config']['boundary_conditions']['acetate']['default']['type'] = 'neumann'

# process 'cell'  (local:CpmCellField)
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['nx'] = 60
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['ny'] = 60
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['seed_block'] = [15, 27, 0, 22, 34, 1]
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['mcs_per_update'] = 8
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['temperature'] = 10.0
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['lambda_volume'] = 2.0
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['contact_j'] = 14.0
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['biomass0'] = 0.1
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['grow_per_biomass'] = 300.0
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['box_volume_L'] = 0.3
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['glucose_km'] = 0.5
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['glucose_vmax'] = 3.5
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['oxygen_vmax'] = 2.5

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: cell-environment-coupling-spatial ===
STUDY = 'cell-environment-coupling-spatial'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**single-cell-in-a-field-movie**


In [ ]:
# single-cell-in-a-field-movie
show_viz(_render_one('image:viz/single-cell-in-a-field.gif', {'chart': 'image', 'caption': 'Glucose (with footprint + COM) | glucose depletion halo (Δ from t=0) | acetate\nplume, over the 20-tick flagship run. The middle panel is the direct niche-construction\nsignal: the cell measurably reshapes the very gradient it depends on.'}, RUNS_DB, STUDY_YAML))

**single-cell-in-a-field-metrics**


In [ ]:
# single-cell-in-a-field-metrics
show_viz(_render_one('html:single-cell-in-a-field-metrics.html', {'chart': 'html', 'caption': 'Synced metrics for the flagship run — local_nutrient / biomass / acetate_secreted\n(left axis) against CPM volume (right axis) over 20 ticks.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cell-metabolizes-and-grows | kind=observable path=obs.biomass expr=last(obs.biomass) | op > value 0.1 provenance over 20 ticks, biomass 0.1 -> ~0.37 (tests/test_flagship_field.py::test_flagship_sense_act_loop; tests/test_cpm_cell_field.py::test_cell_metabolizes_grows_and_reshapes_field asserts the same pattern on a smaller 40x40 grid over 12 ticks) |
| cpm-volume-grows | kind=observable path=obs.volume expr=last(obs.volume) | op > value 40.0 provenance CPM volume 32 -> 110 px over 20 ticks (~3.4x), ~3% of the 60x60=3600px lattice (tests/test_flagship_field.py::test_flagship_sense_act_loop) |
| acetate-secreted-and-diffuses | kind=observable path=fields.acetate expr=sum(fields.acetate) | op > value 0.0 provenance field-wide acetate 0 -> ~47.0 over 20 ticks, spreading into a plume around the cell's footprint (tests/test_flagship_field.py::test_flagship_sense_act_loop; the plume is the third panel of the baked GIF) |
| net-glucose-consumed | kind=observable path=fields.glucose expr=sum(fields.glucose) | op < value 5940.0 provenance field-wide glucose total 5940.0 -> ~5908.4 over 20 ticks (~32 units net consumed); the local footprint drawdown is much sharper (up to ~18%) than the field-wide total because only the cell's footprint and its immediate diffusive neighborhood are touched (tests/test_flagship_field.py::test_flagship_sense_act_loop) |


## Open decisions
- Only 1 of the 9 planned spatial-counterpart studies exists (cell-environment-coupling-spatial, the flagship). The remaining 8 — cellular-interface, cell-cell-coupling, disintegration, molecular-interfaces, biomolecular-complementarity, autopoiesis, growth-and-division, development-and-evolution — are named in the design spec's increment plan but have no composite, study, or run. Treat this investigation's verdict as scoped to the flagship pattern only until the fan-out increments land.
- Chemotaxis toward the sensed field (directed up-gradient motion) is explicitly deferred to a follow-up variant of the flagship composite; the current flagship cell does not chemotax, only senses, metabolizes, grows, and secretes.
